# Latent Dirichlet Allocation (topic modeling)

_Artificial Intelligence — more_

**Every document is a blend of hidden topics. Each topic is a favorite set of words.**

Latent Dirichlet Allocation (LDA) discovers hidden topics in a collection of documents, with no labels.

     Each document is a mixture of topics, described by proportions $\theta$. Each topic is a distribution over words, $\phi$.

---

This notebook is a practice scaffold for the **Latent Dirichlet Allocation (topic modeling)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# vocab 0..3. Docs 0,1 favor words {0,1}; docs 2,3 favor words {2,3}.
docs = [[0,1,0,1], [1,0,1,0], [2,3,2,3], [3,2,3,2]]
V, K = 4, 2                       # vocab size, number of topics
alpha, beta = 0.1, 0.1
z = [rng.integers(0, K, len(d)).tolist() for d in docs]

ndk = np.zeros((len(docs), K))    # doc-topic counts
nkw = np.zeros((K, V))            # topic-word counts
for d, doc in enumerate(docs):
    for i, w in enumerate(doc):
        ndk[d, z[d][i]] += 1; nkw[z[d][i], w] += 1

for _ in range(300):              # collapsed Gibbs sampling
    for d, doc in enumerate(docs):
        for i, w in enumerate(doc):
            k = z[d][i]; ndk[d, k] -= 1; nkw[k, w] -= 1
            p = (ndk[d] + alpha) * (nkw[:, w] + beta) / (nkw.sum(1) + V*beta)
            k = rng.choice(K, p=p / p.sum())
            z[d][i] = k; ndk[d, k] += 1; nkw[k, w] += 1

phi = (nkw + beta) / (nkw.sum(1, keepdims=True) + V*beta)
print("topic-word distributions (rows = topics):")
print(np.round(phi, 2))

## Visualize the data & results

_Six news headlines, two hidden topics: does collapsed-Gibbs LDA separate the Sports words from the Finance words?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1)

# Six real-style news headlines over two topics.
vocab = ["game", "team", "win", "market", "stock", "bank"]
raw = [["team","win","game","game"], ["game","team","win","team"],
       ["win","game","game","team"], ["market","stock","bank","stock"],
       ["stock","market","bank","market"], ["bank","stock","market","stock"]]
wi = {w: i for i, w in enumerate(vocab)}
docs = [[wi[w] for w in h] for h in raw]
V, K, a, b = len(vocab), 2, 0.1, 0.1

z = [rng.integers(0, K, len(d)).tolist() for d in docs]
ndk = np.zeros((len(docs), K)); nkw = np.zeros((K, V))
for d, doc in enumerate(docs):
    for i, w in enumerate(doc):
        ndk[d, z[d][i]] += 1; nkw[z[d][i], w] += 1

for _ in range(500):                     # collapsed Gibbs sampling
    for d, doc in enumerate(docs):
        for i, w in enumerate(doc):
            k = z[d][i]; ndk[d, k] -= 1; nkw[k, w] -= 1
            p = (ndk[d] + a) * (nkw[:, w] + b) / (nkw.sum(1) + V * b)
            k = rng.choice(K, p=p / p.sum())
            z[d][i] = k; ndk[d, k] += 1; nkw[k, w] += 1

phi = (nkw + b) / (nkw.sum(1, keepdims=True) + V * b)
if phi[1, wi["game"]] > phi[0, wi["game"]]:    # order row 0 = Sports
    phi = phi[::-1]

fig, ax = plt.subplots()
im = ax.imshow(phi, cmap="viridis")
for r in range(K):
    for c in range(V):
        ax.text(c, r, round(phi[r, c], 3), ha="center", va="center", color="w")
ax.set_xticks(range(V)); ax.set_xticklabels(vocab)
ax.set_yticks(range(K)); ax.set_yticklabels(["Topic: Sports", "Topic: Finance"])
ax.set_title("Recovered topic-word distributions from 6 news headlines")
fig.colorbar(im, ax=ax)
plt.show()